# 장기/단기 기억 조회 (lookup_db)

챗봇([main.py])이 SQLite 에 저장한 두 종류의 기억을 조회한다.

- **장기기억 (Store, `database/long_term.db`)** — 사용자별 profile · preference · memories
- **단기기억 (Checkpointer, `database/short_term.db`)** — 스레드(대화)별 메시지 히스토리

> 실행 전 `python seed_profile.py` 로 사용자 정보를, `python main.py` 로 대화를 만들어 두면
> 풍부하게 조회할 수 있다. (커널은 프로젝트의 `.venv` 를 선택)

## 0. 준비 — Store / Checkpointer 열기

In [ ]:
import json

from store_setup import create_store, create_checkpointer
from config import (
    PROFILE_KEY,
    PREFERENCE_KEY,
    profile_namespace,
    preferences_namespace,
    memories_namespace,
)

# 챗봇과 동일한 DB 파일에 연결한다.
store = create_store()                 # 장기기억 (long_term.db)
checkpointer = create_checkpointer()   # 단기기억 (short_term.db)
print("store / checkpointer 준비 완료")

## 1. 장기기억(Store) — 저장된 사용자 목록

`("users", <user_id>, <category>)` namespace 들에서 `user_id` 를 추출한다.

In [3]:
# prefix=("users",) 하위의 모든 namespace 조회
namespaces = store.list_namespaces(prefix=("users",))
user_ids = sorted({ns[1] for ns in namespaces if len(ns) >= 2})

print("저장된 사용자:", user_ids)
print("\n전체 namespace:")
for ns in namespaces:
    print("  ", ns)

저장된 사용자: ['demo_user', 'james', 'minji']

전체 namespace:
   ('users', 'demo_user', 'memories')
   ('users', 'demo_user', 'preferences')
   ('users', 'demo_user', 'profile')
   ('users', 'james', 'preferences')
   ('users', 'james', 'profile')
   ('users', 'minji', 'preferences')
   ('users', 'minji', 'profile')


## 2. 사용자별 profile · preference · memories

- **profile / preference**: 고정 key(`basic_info` / `response_style`) → `store.get()`
- **memories**: 동적 UUID key 누적 → `store.search(query=None)` 로 전체 나열

In [4]:
def show_user(user_id: str) -> None:
    """한 사용자의 장기기억(profile/preference/memories)을 출력한다."""
    print("=" * 64)
    print(f"[user_id = {user_id}]")

    # profile (고정 key)
    profile = store.get(profile_namespace(user_id), PROFILE_KEY)
    print("  ● profile   :", json.dumps(profile.value, ensure_ascii=False) if profile else None)

    # preference (고정 key)
    pref = store.get(preferences_namespace(user_id), PREFERENCE_KEY)
    print("  ● preference:", json.dumps(pref.value, ensure_ascii=False) if pref else None)

    # memories (동적 key 누적). query 없이 호출하면 전체를 나열한다.
    memories = store.search(memories_namespace(user_id), limit=100)
    print(f"  ● memories  : {len(memories)}건")
    for item in memories:
        print(f"      - {item.value.get('content', '')}   (key={item.key[:8]}…)")


for uid in user_ids:
    show_user(uid)

[user_id = demo_user]
  ● profile   : {"name": "홍길동", "job": "AI 개발자", "language": "Korean", "timezone": "Asia/Seoul"}
  ● preference: {"language": "English", "tone": "clear", "format": "markdown"}
  ● memories  : 4건
      - 사용자는 일주일 전에 제주도 한라산 등산을 다녀왔고 매우 만족했다.   (key=46beb0d9…)
      - 사용자는 한 달 전에 독산동으로 이사왔다.   (key=8c1ded2c…)
      - 사용자는 파이썬 장고에 관심이 많고 현재 장고 프로젝트를 진행 중이다.   (key=84e848cf…)
      - 사용자는 웹 개발자이다.   (key=c3d5dfa1…)
[user_id = james]
  ● profile   : {"name": "James Park", "job": "데이터 분석가", "language": "English", "timezone": "America/New_York"}
  ● preference: {"language": "English", "tone": "concise", "format": "json"}
  ● memories  : 0건
[user_id = minji]
  ● profile   : {"name": "김민지", "job": "백엔드 개발자", "language": "Korean", "timezone": "Asia/Seoul"}
  ● preference: {"language": "Korean", "tone": "friendly", "format": "text"}
  ● memories  : 0건


## 3. memories 의미 기반 검색 (semantic search)

Store 의 임베딩 `index` 를 이용해 질문과 **의미적으로 유사한** 기억을 찾는다.
(점수가 클수록 질의와 유사. `save_memory` Tool 이 저장한 기억이 대상)

In [7]:
target_user = "demo_user"
query = "제주도 관련 기억"   # 자유롭게 바꿔보세요. 예) "취미", "사는 곳"
query = "현재 사는 곳"
results = store.search(memories_namespace(target_user), query=query, limit=5)
print(f"[{target_user}] '{query}' 검색 결과 ({len(results)}건):")
for item in results:
    score = getattr(item, "score", None)
    print(f"  - {item.value.get('content', '')}   (score={score:.3f})" if score is not None
          else f"  - {item.value.get('content', '')}")

[demo_user] '현재 사는 곳' 검색 결과 (4건):
  - 사용자는 파이썬 장고에 관심이 많고 현재 장고 프로젝트를 진행 중이다.   (score=0.267)
  - 사용자는 한 달 전에 독산동으로 이사왔다.   (score=0.250)
  - 사용자는 웹 개발자이다.   (score=0.224)
  - 사용자는 일주일 전에 제주도 한라산 등산을 다녀왔고 매우 만족했다.   (score=0.224)


## 4. 단기기억(Checkpointer) — 대화 스레드 목록

체크포인터에는 `thread_id` 별로 대화 상태가 쌓인다. 전체를 훑어 스레드를 모은다.

In [8]:
# config=None 으로 list 하면 모든 스레드의 체크포인트를 순회한다.
thread_ids = []
for ct in checkpointer.list(None):
    tid = ct.config["configurable"]["thread_id"]
    if tid not in thread_ids:
        thread_ids.append(tid)

print("대화 스레드(thread_id):", thread_ids if thread_ids
      else "아직 대화 기록 없음 — `python main.py` 로 대화 후 다시 실행하세요.")

대화 스레드(thread_id): ['1']


## 5. 특정 스레드의 대화 히스토리

해당 스레드의 **최신 체크포인트**에 저장된 메시지 목록을 보여준다.
(AIMessage 의 도구 호출도 함께 표시)

In [9]:
def show_thread(thread_id: str) -> None:
    """한 스레드의 최신 메시지 히스토리를 출력한다."""
    cfg = {"configurable": {"thread_id": thread_id}}
    snapshot = checkpointer.get_tuple(cfg)
    if snapshot is None:
        print(f"thread '{thread_id}' : 기록 없음")
        return

    messages = snapshot.checkpoint["channel_values"].get("messages", [])
    print(f"[thread_id = {thread_id}] 메시지 {len(messages)}개")
    for m in messages:
        role = type(m).__name__.replace("Message", "")
        content = (m.content or "").strip()
        print(f"\n[{role}] {content}".rstrip())
        # AIMessage 가 호출한 Tool 표시
        for tc in getattr(m, "tool_calls", []) or []:
            print(f"    ↳ tool_call: {tc['name']}({tc['args']})")


if thread_ids:
    show_thread(thread_ids[0])   # 첫 스레드. 다른 스레드는 인자를 바꿔 호출.
else:
    print("표시할 스레드가 없습니다.")

[thread_id = 1] 메시지 26개

[Human] 안녕하세요.

[AI] 안녕하세요, 홍길동님.  
무엇을 도와드릴까요?

[Human] 나는 웹 개발자야.

[AI]
    ↳ tool_call: save_memory({'content': '사용자는 웹 개발자이다.'})

[Tool] 기억을 저장했습니다: 사용자는 웹 개발자이다.

[AI] 알겠습니다.  
`웹 개발자`로 기억해둘게요.

원하시면 프론트엔드/백엔드/풀스택 중 더 구체적으로도 저장해드릴 수 있어요.

[Human] 나는 파이썬 장고에 관심이 많아. 현재 장고 프로젝트를 하고있어.

[AI]
    ↳ tool_call: save_memory({'content': '사용자는 파이썬 장고에 관심이 많고 현재 장고 프로젝트를 진행 중이다.'})

[Tool] 기억을 저장했습니다: 사용자는 파이썬 장고에 관심이 많고 현재 장고 프로젝트를 진행 중이다.

[AI] 좋아요.  
`파이썬 장고에 관심이 많고 현재 장고 프로젝트를 진행 중`인 것으로 기억해둘게요.

장고 관련해서 모델, 뷰, 템플릿, ORM, 배포, 성능 최적화 등 뭐든 도와드릴 수 있어요.

[Human] 나는 한달 전에 독산동으로 이사왔어.

[AI]
    ↳ tool_call: save_memory({'content': '사용자는 한 달 전에 독산동으로 이사왔다.'})

[Tool] 기억을 저장했습니다: 사용자는 한 달 전에 독산동으로 이사왔다.

[AI] 알겠습니다.  
`한 달 전에 독산동으로 이사왔다`는 점도 기억해둘게요.

필요하시면 독산동 주변 생활 정보나 추천 장소도 함께 찾아드릴 수 있어요.

[Human] 일주일 전에 제주도 한라산 등산을 다녀왔어. 너무 좋았어.

[AI]
    ↳ tool_call: save_memory({'content': '사용자는 일주일 전에 제주도 한라산 등산을 다녀왔고 매우 만족했다.'})

[Tool] 기억을 저장했습니다: 사용자는 일주일 전에 제주도 한라산 등산을 다녀왔고 매우

## 6. (참고) 스레드별 체크포인트 개수

한 번의 대화 turn 안에서도 그래프의 super-step 마다 체크포인트가 누적된다.

In [10]:
for tid in thread_ids:
    n = len(list(checkpointer.list({"configurable": {"thread_id": tid}})))
    print(f"thread '{tid}': 체크포인트 {n}개")
if not thread_ids:
    print("체크포인트 없음")

thread '1': 체크포인트 40개
